## GA Parameter Tuning

The point of this notebook is to tune the parameters of the genetic algorithm. Only mutation probability and warm start ratio are tuned here for time efficiency. Since this is a study and not a productionized pipeline, a faster more brief tuning is enough. Naturally there is also an attempt to increase time complexity to see if it helps. The metric is `'central_tail_effectiveness'` due to the fact that expected shortfall is expected to be the most difficult to optimize and the central variant is less prone to invalid values. Tuning is performed on the second half of validation data (2019).

## Setup

In [1]:
import itertools
import os
from pathlib import Path

import pandas as pd
from joblib import Parallel, delayed

In [2]:
# Find project root (folder that contains .git)
ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent

# Set working directory to root
os.chdir(ROOT)

print("Now working in:", Path.cwd())

Now working in: C:\Users\couch\OneDrive\Assignments\Master Thesis\Repo


In [3]:
from src.config import TEST_MODELS

In [4]:
MUTATION_PROBABILITIES = [0.05, 0.2]
WARM_START_RATIOS = [0.2, 0.5]

NGENS = [40, 80]
POP_SIZES = [100, 200]

## Tuning Mutation and Warm Start

In [5]:
def run_single_combination(params):
    import os
    import sys
    from pathlib import Path

    import pandas as pd

    # Find project root (folder that contains .git)
    ROOT = Path.cwd()
    while not (ROOT / ".git").exists():
        ROOT = ROOT.parent

    # Set working directory AND update Python's import path
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

    # Suppress all tqdm progress bars in this process
    os.environ["TQDM_DISABLE"] = "1"

    from src.config import TUNE_END, TUNE_START, WINDOW_SIZE
    from src.simulation import run_backtest

    mutpb, warm_start_ratio, model_tuple = params
    mean_model, volatility_model, _ = model_tuple

    results_dir = Path("results/ga_tuning/mutation_warm_start")

    m_prob_str = str(mutpb).replace(".", "p")
    ws_ratio_str = str(warm_start_ratio).replace(".", "p")

    combo_name = f"mprob_{m_prob_str}_wsr_{ws_ratio_str}"
    combo_dir = results_dir / combo_name
    data_dir = combo_dir / "data"
    logs_dir = combo_dir / "logs"

    data_dir.mkdir(parents=True, exist_ok=True)
    logs_dir.mkdir(parents=True, exist_ok=True)

    result_file = data_dir / f"{mean_model}_{volatility_model}.parquet"
    log_file = logs_dir / f"{mean_model}_{volatility_model}.txt"

    if result_file.exists():
        return

    df = pd.read_parquet("results/processed_data/stock_data.parquet", engine="pyarrow")
    df.index = pd.to_datetime(df.index)
    df = df[df.index <= TUNE_END]

    run_backtest(
        df,
        window_size=WINDOW_SIZE,
        start_date=TUNE_START,
        end_date=TUNE_END,
        mean_model=mean_model,
        volatility_model=volatility_model,
        optimize_portfolio_flag=True,
        save_path=result_file,
        ga_metric="central_tail_effectiveness",
        rf_rates_path="results/processed_data/risk_free_rate.json",
        log_path=log_file,
        ngen=40,
        pop_size=100,
        mutpb=mutpb,
        warm_start_ratio=warm_start_ratio,
    )

    print(f"Finished ({mean_model}, {volatility_model}) for ({mutpb}, {warm_start_ratio})", flush=True)


param_combinations = list(itertools.product(MUTATION_PROBABILITIES, WARM_START_RATIOS, TEST_MODELS))

Parallel(n_jobs=4, backend="loky", batch_size=1)(delayed(run_single_combination)(p) for p in param_combinations)
print("Finished backtest for all parameters")

Finished backtest for all parameters


In [6]:
results_dir = Path("results/ga_tuning/mutation_warm_start")
records = []

# Iterate over all parquet files matching the runner's path structure
for parquet_file in results_dir.glob("mprob_*/data/*.parquet"):
    # Extract parameter folder (e.g., 'mprob_0p05_wsr_0p2')
    combo_folder = parquet_file.parent.parent.name
    parts = combo_folder.split("_")

    mutpb = float(parts[1].replace("p", "."))
    warm_start_ratio = float(parts[3].replace("p", "."))
    model_stem = parquet_file.stem

    if model_stem.startswith("naive"):
        model_name = "Brak modelu"
    else:
        model_name = model_stem.split("_")[-1].upper()

    df = pd.read_parquet(parquet_file, engine="pyarrow")

    # Check for non-finite flags (False values) in ga_metric_is_finite
    warning_mask = (~df["ga_metric_is_finite"]) & df["best_ga_metric_value"].notna()
    if warning_mask.any():
        count = warning_mask.sum()
        print(f"[WARNING] Non-finite GA metric with non-NaN value detected in {parquet_file} ({count} occurrence(s))")

    # Compute overall metric value for this combination
    metric_val = df["best_ga_metric_value"].mean()

    records.append(
        {
            "Pr. mutacji": mutpb,
            "Prop. z poprzedniego dnia": warm_start_ratio,
            "Model": model_name,
            "metric_value": metric_val,
        }
    )

df_all = pd.DataFrame(records)

# Pivot models into individual columns
summary_df_mprop_wstart = df_all.pivot(
    index=["Pr. mutacji", "Prop. z poprzedniego dnia"],
    columns="Model",
    values="metric_value",
)

# Extract model column names and calculate row-wise average
summary_df_mprop_wstart["Średnia"] = summary_df_mprop_wstart.mean(axis=1)

# Format numbers: round to 4 decimal places and replace '.' with ','
for col in summary_df_mprop_wstart.columns:
    summary_df_mprop_wstart[col] = summary_df_mprop_wstart[col].apply(lambda x: f"{x:.6f}".replace(".", ","))
display(summary_df_mprop_wstart)

print(summary_df_mprop_wstart.to_latex(index=True))

Model                                 Brak modelu       CCC       DCC  \
Pr. mutacji Prop. z poprzedniego dnia                                   
0.05        0.2                          0,052152  0,056166  0,056113   
            0.5                          0,052143  0,056129  0,056080   
0.20        0.2                          0,052173  0,056184  0,056139   
            0.5                          0,052172  0,056172  0,056129   

Model                                     GARCH     NAIVE   Średnia  
Pr. mutacji Prop. z poprzedniego dnia                                
0.05        0.2                        0,052230  0,050498  0,053432  
            0.5                        0,052162  0,050476  0,053398  
0.20        0.2                        0,052266  0,050528  0,053458  
            0.5                        0,052238  0,050522  0,053447

\begin{tabular}{llllllll}
\toprule
 & Model & Brak modelu & CCC & DCC & GARCH & NAIVE & Średnia \\
Pr. mutacji & Prop. z poprzedniego dnia &  &  &  &  &  &  \\
\midrule
\multirow[t]{2}{*}{0.050000} & 0.200000 & 0,052152 & 0,056166 & 0,056113 & 0,052230 & 0,050498 & 0,053432 \\
 & 0.500000 & 0,052143 & 0,056129 & 0,056080 & 0,052162 & 0,050476 & 0,053398 \\
\cline{1-8}
\multirow[t]{2}{*}{0.200000} & 0.200000 & 0,052173 & 0,056184 & 0,056139 & 0,052266 & 0,050528 & 0,053458 \\
 & 0.500000 & 0,052172 & 0,056172 & 0,056129 & 0,052238 & 0,050522 & 0,053447 \\
\cline{1-8}
\bottomrule
\end{tabular}



## Tuning Time Complexity

In [7]:
def run_single_combination(params):
    import os
    import sys
    from pathlib import Path

    import pandas as pd

    # Find project root (folder that contains .git)
    ROOT = Path.cwd()
    while not (ROOT / ".git").exists():
        ROOT = ROOT.parent

    # Set working directory AND update Python's import path
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

    # Suppress all tqdm progress bars in this process
    os.environ["TQDM_DISABLE"] = "1"

    from src.config import TUNE_END, TUNE_START, WINDOW_SIZE
    from src.simulation import run_backtest

    ngen, pop_size, model_tuple = params
    mean_model, volatility_model, _ = model_tuple

    results_dir = Path("results/ga_tuning/time_complexity")

    combo_name = f"ngen_{ngen}_popsize_{pop_size}"
    combo_dir = results_dir / combo_name
    data_dir = combo_dir / "data"
    logs_dir = combo_dir / "logs"

    data_dir.mkdir(parents=True, exist_ok=True)
    logs_dir.mkdir(parents=True, exist_ok=True)

    result_file = data_dir / f"{mean_model}_{volatility_model}.parquet"
    log_file = logs_dir / f"{mean_model}_{volatility_model}.txt"

    if result_file.exists():
        return

    df = pd.read_parquet("results/processed_data/stock_data.parquet", engine="pyarrow")
    df.index = pd.to_datetime(df.index)
    df = df[df.index <= TUNE_END]

    run_backtest(
        df,
        window_size=WINDOW_SIZE,
        start_date=TUNE_START,
        end_date=TUNE_END,
        mean_model=mean_model,
        volatility_model=volatility_model,
        optimize_portfolio_flag=True,
        save_path=result_file,
        ga_metric="central_tail_effectiveness",
        rf_rates_path="results/processed_data/risk_free_rate.json",
        log_path=log_file,
        ngen=ngen,
        pop_size=pop_size,
        mutpb=0.2,
        warm_start_ratio=0.2,
    )

    print(f"Finished ({mean_model}, {volatility_model}) for ({ngen}, {pop_size})", flush=True)


param_combinations = list(itertools.product(NGENS, POP_SIZES, TEST_MODELS))

Parallel(n_jobs=4, backend="loky", batch_size=1)(delayed(run_single_combination)(p) for p in param_combinations)
print("Finished backtest for all parameters")

Finished backtest for all parameters


In [8]:
results_dir = Path("results/ga_tuning/time_complexity")
records = []

# Iterate over all parquet files matching the runner's path structure
for parquet_file in results_dir.glob("ngen_*/data/*.parquet"):
    combo_folder = parquet_file.parent.parent.name
    parts = combo_folder.split("_")

    ngen = int(parts[1])
    pop_size = int(parts[3])
    model_stem = parquet_file.stem

    if model_stem.startswith("naive"):
        model_name = "Brak modelu"
    else:
        model_name = model_stem.split("_")[-1].upper()

    df = pd.read_parquet(parquet_file, engine="pyarrow")

    # Check for non-finite flags (False values) in ga_metric_is_finite
    warning_mask = (~df["ga_metric_is_finite"]) & df["best_ga_metric_value"].notna()
    if warning_mask.any():
        count = warning_mask.sum()
        print(f"[WARNING] Non-finite GA metric with non-NaN value detected in {parquet_file} ({count} occurrence(s))")

    # Compute overall metric value for this combination
    metric_val = df["best_ga_metric_value"].mean()

    records.append(
        {
            "Generacje": ngen,
            "Populacja": pop_size,
            "Model": model_name,
            "metric_value": metric_val,
        }
    )

df_all = pd.DataFrame(records)

# Pivot models into individual columns
summary_df_time_comp = df_all.pivot(
    index=["Generacje", "Populacja"],
    columns="Model",
    values="metric_value",
)

# Extract model column names and calculate row-wise average
summary_df_time_comp["Średnia"] = summary_df_time_comp.mean(axis=1)

# Format numbers: round to 4 decimal places and replace '.' with ','
for col in summary_df_time_comp.columns:
    summary_df_time_comp[col] = summary_df_time_comp[col].apply(lambda x: f"{x:.6f}".replace(".", ","))
display(summary_df_time_comp)

print(summary_df_time_comp.to_latex(index=True))

Model               Brak modelu       CCC       DCC     GARCH     NAIVE  \
Generacje Populacja                                                       
40        100          0,052173  0,056188  0,056139  0,052266  0,050528   
          200          0,052180  0,056199  0,056148  0,052274  0,050531   
80        100          0,052188  0,056203  0,056156  0,052283  0,050540   
          200          0,052190  0,056234  0,056158  0,052286  0,050541   

Model                 Średnia  
Generacje Populacja            
40        100        0,053459  
          200        0,053466  
80        100        0,053474  
          200        0,053482

\begin{tabular}{llllllll}
\toprule
 & Model & Brak modelu & CCC & DCC & GARCH & NAIVE & Średnia \\
Generacje & Populacja &  &  &  &  &  &  \\
\midrule
\multirow[t]{2}{*}{40} & 100 & 0,052173 & 0,056188 & 0,056139 & 0,052266 & 0,050528 & 0,053459 \\
 & 200 & 0,052180 & 0,056199 & 0,056148 & 0,052274 & 0,050531 & 0,053466 \\
\cline{1-8}
\multirow[t]{2}{*}{80} & 100 & 0,052188 & 0,056203 & 0,056156 & 0,052283 & 0,050540 & 0,053474 \\
 & 200 & 0,052190 & 0,056234 & 0,056158 & 0,052286 & 0,050541 & 0,053482 \\
\cline{1-8}
\bottomrule
\end{tabular}

